# Human Activity Recognition Using Hidden Markov Models

This notebook implements a complete Hidden Markov Model pipeline for classifying human physical activities from accelerometer and gyroscope sensor data.

## Problem Overview

We aim to recognize four distinct human activities:
- **Standing**: Stationary position at waist level
- **Walking**: Continuous walking motion with consistent pace
- **Jumping**: Repeated jumping motion
- **Still**: Stationary position on flat surface

## Methodology

1. **Feature Extraction**: Extract time-domain and frequency-domain features from sensor windows
2. **Model Architecture**: Implement Gaussian Hidden Markov Model with four states
3. **Training**: Fit HMM parameters using labeled training data
4. **Inference**: Use Viterbi algorithm to decode most likely activity sequences
5. **Evaluation**: Assess model performance on test data and analyze results

## 1. Environment Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import os
import warnings

warnings.filterwarnings('ignore')

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Set random seeds for reproducibility
np.random.seed(42)

print("Environment setup complete.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Data Loading and Exploration

In [ ]:
# Data directory configuration
DATA_DIR = '../data'

# Activity mapping: (pattern, label, display_name)
ACTIVITIES = {
    'standing_waist': (0, 'Standing'),
    'walking': (1, 'Walking'),
    'jumping': (2, 'Jumping'),
    'still': (3, 'Still')
}

def load_activity_data(directory, activity_pattern):
    """
    Load all CSV files matching a pattern from a directory.
    
    Parameters
    ----------
    directory : str
        Directory containing data files
    activity_pattern : str
        Pattern to match filenames
        
    Returns
    -------
    pandas.DataFrame
        Aggregated data from all matching files
    """
    dataframes = []
    
    for filename in os.listdir(directory):
        if activity_pattern in filename and filename.endswith('.csv'):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath)
            dataframes.append(df)
    
    if not dataframes:
        raise ValueError(f"No files found matching pattern '{activity_pattern}'")
    
    return pd.concat(dataframes, ignore_index=True)

def standardize_column_names(df):
    """
    Standardize column names to consistent format.
    """
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
    
    column_mapping = {
        'accelx': 'accel_x', 'accely': 'accel_y', 'accelz': 'accel_z',
        'gyrox': 'gyro_x', 'gyroy': 'gyro_y', 'gyroz': 'gyro_z',
        'time': 'timestamp', 'timestamp_ms': 'timestamp'
    }
    
    df.rename(columns=column_mapping, inplace=True)
    return df

# Load all activity data
print("Loading activity data...")
data_list = []
label_list = []
activity_names = {}

for pattern, (label, name) in sorted(ACTIVITIES.items()):
    try:
        data = load_activity_data(DATA_DIR, pattern)
        data = standardize_column_names(data)
        data_list.append(data)
        label_list.append(np.full(len(data), label))
        activity_names[label] = name
        print(f"  {name}: {len(data)} samples loaded")
    except ValueError as e:
        print(f"  Warning: {e}")

print(f"\nTotal samples loaded: {sum(len(d) for d in data_list)}")

In [ ]:
# Display sample statistics
print("\nData Summary")
print("=" * 60)

for i, (data, label) in enumerate(zip(data_list, label_list)):
    activity_name = activity_names[label[0]]
    print(f"\n{activity_name}:")
    print(f"  Shape: {data.shape}")
    print(f"  Columns: {list(data.columns)}")
    print(f"  Unique activity label: {np.unique(label)[0]}")

# Display sample data
print("\n" + "=" * 60)
print("Sample data (first 5 rows of Standing activity)")
print("=" * 60)
print(data_list[0].head())

## 3. Feature Extraction

In [ ]:
class FeatureExtractor:
    """
    Extract time-domain and frequency-domain features from sensor signals.
    """
    
    WINDOW_SIZE = 50
    OVERLAP = 25
    
    @staticmethod
    def extract_time_domain_features(signal_data):
        """
        Extract statistical features from signal window.
        """
        signal_data = np.asarray(signal_data, dtype=np.float64)
        
        return {
            'mean': np.mean(signal_data),
            'std': np.std(signal_data),
            'variance': np.var(signal_data),
            'min': np.min(signal_data),
            'max': np.max(signal_data),
            'range': np.max(signal_data) - np.min(signal_data),
            'rms': np.sqrt(np.mean(signal_data ** 2)),
            'mad': np.mean(np.abs(signal_data - np.mean(signal_data))),
        }
    
    @staticmethod
    def extract_frequency_domain_features(signal_data, sampling_rate=50):
        """
        Extract spectral features using FFT.
        """
        signal_data = np.asarray(signal_data, dtype=np.float64)
        
        fft_values = np.abs(np.fft.fft(signal_data))
        fft_freq = np.fft.fftfreq(len(signal_data), 1/sampling_rate)
        
        positive_freq_idx = fft_freq > 0
        fft_values = fft_values[positive_freq_idx]
        fft_freq = fft_freq[positive_freq_idx]
        
        dominant_freq_idx = np.argmax(fft_values)
        dominant_freq = fft_freq[dominant_freq_idx] if len(fft_freq) > 0 else 0
        
        return {
            'spectral_energy': np.sum(fft_values ** 2),
            'spectral_entropy': -np.sum((fft_values / (np.sum(fft_values) + 1e-10)) ** 2 + 1e-10),
            'dominant_frequency': dominant_freq,
            'frequency_centroid': np.sum(fft_freq * fft_values) / (np.sum(fft_values) + 1e-10),
        }
    
    @staticmethod
    def compute_signal_magnitude_area(accel_x, accel_y, accel_z):
        """
        Compute signal magnitude area from 3-axis accelerometer data.
        """
        accel_x = np.asarray(accel_x, dtype=np.float64)
        accel_y = np.asarray(accel_y, dtype=np.float64)
        accel_z = np.asarray(accel_z, dtype=np.float64)
        
        magnitude = np.sqrt(accel_x ** 2 + accel_y ** 2 + accel_z ** 2)
        return np.mean(magnitude)
    
    @staticmethod
    def compute_axis_correlation(signal_data_1, signal_data_2):
        """
        Compute correlation between two signal axes.
        """
        signal_data_1 = np.asarray(signal_data_1, dtype=np.float64)
        signal_data_2 = np.asarray(signal_data_2, dtype=np.float64)
        
        correlation = np.corrcoef(signal_data_1, signal_data_2)[0, 1]
        return correlation if not np.isnan(correlation) else 0.0
    
    @classmethod
    def extract_window_features(cls, window_data, sampling_rate=50):
        """
        Extract all features from a sensor data window.
        """
        features = {}
        
        # Accelerometer features
        for axis in ['x', 'y', 'z']:
            col_name = f'accel_{axis}'
            if col_name in window_data.columns:
                signal_data = window_data[col_name].values
                time_features = cls.extract_time_domain_features(signal_data)
                freq_features = cls.extract_frequency_domain_features(signal_data, sampling_rate)
                
                for key, value in time_features.items():
                    features[f'accel_{axis}_time_{key}'] = value
                for key, value in freq_features.items():
                    features[f'accel_{axis}_freq_{key}'] = value
        
        # Gyroscope features
        for axis in ['x', 'y', 'z']:
            col_name = f'gyro_{axis}'
            if col_name in window_data.columns:
                signal_data = window_data[col_name].values
                time_features = cls.extract_time_domain_features(signal_data)
                freq_features = cls.extract_frequency_domain_features(signal_data, sampling_rate)
                
                for key, value in time_features.items():
                    features[f'gyro_{axis}_time_{key}'] = value
                for key, value in freq_features.items():
                    features[f'gyro_{axis}_freq_{key}'] = value
        
        # Multi-axis features
        if all(col in window_data.columns for col in ['accel_x', 'accel_y', 'accel_z']):
            sma = cls.compute_signal_magnitude_area(
                window_data['accel_x'].values,
                window_data['accel_y'].values,
                window_data['accel_z'].values
            )
            features['accel_sma'] = sma
            
            features['accel_xy_corr'] = cls.compute_axis_correlation(
                window_data['accel_x'].values, window_data['accel_y'].values
            )
            features['accel_xz_corr'] = cls.compute_axis_correlation(
                window_data['accel_x'].values, window_data['accel_z'].values
            )
            features['accel_yz_corr'] = cls.compute_axis_correlation(
                window_data['accel_y'].values, window_data['accel_z'].values
            )
        
        return features
    
    @classmethod
    def extract_from_dataframe(cls, df, sampling_rate=50, window_size=None, overlap=None):
        """
        Extract features from entire dataframe using sliding windows.
        """
        window_size = window_size or cls.WINDOW_SIZE
        overlap = overlap or cls.OVERLAP
        stride = window_size - overlap
        
        feature_list = []
        
        for start_idx in range(0, len(df) - window_size + 1, stride):
            end_idx = start_idx + window_size
            window = df.iloc[start_idx:end_idx]
            
            features = cls.extract_window_features(window, sampling_rate)
            feature_list.append(features)
        
        return pd.DataFrame(feature_list)

print("Feature extraction class defined.")

In [ ]:
# Extract features from all activities
print("Extracting features...\n")

feature_list = []
labels_all = []

for data, labels in zip(data_list, label_list):
    activity_name = activity_names[labels[0]]
    print(f"Processing {activity_name}...", end=" ")
    
    features = FeatureExtractor.extract_from_dataframe(data)
    feature_list.append(features)
    
    # Create corresponding labels (one per feature window)
    activity_labels = np.full(len(features), labels[0])
    labels_all.append(activity_labels)
    
    print(f"Generated {len(features)} feature windows")

# Combine all features and labels
X = pd.concat(feature_list, ignore_index=True).values
y = np.concatenate(labels_all)

print(f"\nTotal features extracted: {X.shape}")
print(f"Feature dimension: {X.shape[1]}")
print(f"\nLabel distribution:")
for label in sorted(np.unique(y)):
    count = np.sum(y == label)
    percentage = 100 * count / len(y)
    print(f"  {activity_names[label]}: {count} samples ({percentage:.1f}%)")

## 4. Data Preprocessing and Train-Test Split

In [ ]:
# Normalize features
print("Normalizing features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nData split completed:")
print(f"  Training set: {X_train.shape[0]} samples")
print(f"  Test set: {X_test.shape[0]} samples")
print(f"\nTraining set label distribution:")
for label in sorted(np.unique(y_train)):
    count = np.sum(y_train == label)
    percentage = 100 * count / len(y_train)
    print(f"  {activity_names[label]}: {count} ({percentage:.1f}%)")

print(f"\nTest set label distribution:")
for label in sorted(np.unique(y_test)):
    count = np.sum(y_test == label)
    percentage = 100 * count / len(y_test)
    print(f"  {activity_names[label]}: {count} ({percentage:.1f}%)")

## 5. Hidden Markov Model Implementation

In [ ]:
class GaussianHMM:
    """
    Gaussian Hidden Markov Model for activity sequence classification.
    
    Uses Viterbi algorithm for decoding and supports training from labeled data.
    """
    
    def __init__(self, n_states=4, random_state=None):
        """
        Initialize HMM parameters.
        
        Parameters
        ----------
        n_states : int
            Number of hidden states (activities)
        random_state : int, optional
            Random seed for reproducibility
        """
        self.n_states = n_states
        self.random_state = random_state
        
        if random_state is not None:
            np.random.seed(random_state)
        
        self.means_ = None
        self.covariances_ = None
        self.weights_ = None
        self.transmat_ = None
        self.n_features = None
    
    def _initialize_from_data(self, X, y):
        """
        Initialize model parameters from labeled training data.
        """
        self.n_features = X.shape[1]
        self.means_ = np.zeros((self.n_states, self.n_features))
        self.covariances_ = np.zeros((self.n_states, self.n_features, self.n_features))
        
        # Estimate Gaussian parameters for each state
        for state in range(self.n_states):
            mask = y == state
            if np.sum(mask) > 1:
                self.means_[state] = np.mean(X[mask], axis=0)
                self.covariances_[state] = np.cov(X[mask].T)
            else:
                self.means_[state] = np.mean(X, axis=0)
                self.covariances_[state] = np.eye(self.n_features)
            
            # Ensure positive definiteness
            self.covariances_[state] += np.eye(self.n_features) * 1e-6
        
        self._initialize_transition_matrix(y)
        
        # Initialize state priors
        self.weights_ = np.zeros(self.n_states)
        for state in range(self.n_states):
            self.weights_[state] = np.sum(y == state) / len(y)
    
    def _initialize_transition_matrix(self, y):
        """
        Initialize transition matrix from labeled sequences.
        """
        transmat = np.ones((self.n_states, self.n_states)) + 1e-6
        
        for t in range(len(y) - 1):
            from_state = int(y[t])
            to_state = int(y[t + 1])
            transmat[from_state, to_state] += 1
        
        self.transmat_ = transmat / transmat.sum(axis=1, keepdims=True)
    
    def _compute_emission_probability(self, obs, state):
        """
        Compute emission probability density for observation given state.
        """
        mean = self.means_[state]
        cov = self.covariances_[state]
        
        diff = obs - mean
        inv_cov = np.linalg.inv(cov)
        cov_det = np.linalg.det(cov)
        
        normalization = 1.0 / np.sqrt((2 * np.pi) ** self.n_features * cov_det)
        exponent = -0.5 * diff @ inv_cov @ diff.T
        
        return normalization * np.exp(exponent)
    
    def fit(self, X, y):
        """
        Train HMM using labeled data.
        """
        self._initialize_from_data(X, y)
        return self
    
    def viterbi_decode(self, observations):
        """
        Decode most likely state sequence using Viterbi algorithm.
        
        Returns
        -------
        path : ndarray
            Most likely state sequence
        log_prob : float
            Log probability of the sequence
        """
        n_obs = len(observations)
        
        viterbi = np.zeros((n_obs, self.n_states))
        backpointer = np.zeros((n_obs, self.n_states), dtype=int)
        
        # Initialize first timestep
        for state in range(self.n_states):
            emission = self._compute_emission_probability(observations[0], state)
            viterbi[0, state] = np.log(self.weights_[state] + 1e-10) + np.log(emission + 1e-10)
        
        # Forward pass
        for t in range(1, n_obs):
            for state in range(self.n_states):
                emission = self._compute_emission_probability(observations[t], state)
                
                trans_probs = np.log(self.transmat_[:, state] + 1e-10)
                prev_probs = viterbi[t-1] + trans_probs
                
                backpointer[t, state] = np.argmax(prev_probs)
                viterbi[t, state] = np.max(prev_probs) + np.log(emission + 1e-10)
        
        # Backtrack
        path = np.zeros(n_obs, dtype=int)
        path[-1] = np.argmax(viterbi[-1])
        
        for t in range(n_obs - 2, -1, -1):
            path[t] = backpointer[t + 1, int(path[t + 1])]
        
        log_prob = np.max(viterbi[-1])
        
        return path, log_prob
    
    def predict(self, X):
        """
        Predict activity labels for feature vectors.
        """
        predictions = np.zeros(len(X), dtype=int)
        
        for i, obs in enumerate(X):
            path, _ = self.viterbi_decode(X[max(0, i-10):i+1])
            predictions[i] = path[-1]
        
        return predictions
    
    def score(self, X, y):
        """
        Compute accuracy score on test data.
        """
        predictions = self.predict(X)
        return accuracy_score(y, predictions)

print("Gaussian HMM class defined.")

## 6. Model Training

In [ ]:
# Train the HMM model
print("Training HMM model...\n")

hmm_model = GaussianHMM(n_states=4, random_state=42)
hmm_model.fit(X_train, y_train)

print("Model training completed.")
print(f"\nModel Parameters:")
print(f"  Number of states: {hmm_model.n_states}")
print(f"  Feature dimension: {hmm_model.n_features}")
print(f"  State prior probabilities: {np.round(hmm_model.weights_, 4)}")

print(f"\nTransition Matrix (from state i to state j):")
state_labels = [activity_names[i] for i in range(4)]
transmat_df = pd.DataFrame(
    hmm_model.transmat_,
    index=[f"{s} (from)" for s in state_labels],
    columns=[f"{s} (to)" for s in state_labels]
)
print(transmat_df.round(4))

## 7. Model Evaluation

In [ ]:
# Predictions on test set
print("Generating predictions on test set...")
y_pred = hmm_model.predict(X_test)

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average=None, zero_division=0)
recall = recall_score(y_test, y_pred, average=None, zero_division=0)
f1 = f1_score(y_test, y_pred, average=None, zero_division=0)

print(f"\nOverall Model Performance:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Macro Precision: {np.mean(precision):.4f}")
print(f"  Macro Recall: {np.mean(recall):.4f}")
print(f"  Macro F1-Score: {np.mean(f1):.4f}")

# Per-class metrics
print(f"\nPer-Activity Metrics:")
print(f"{'Activity':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Samples':<10}")
print("-" * 60)

for label in range(4):
    activity_name = activity_names[label]
    n_samples = np.sum(y_test == label)
    print(f"{activity_name:<15} {precision[label]:<12.4f} {recall[label]:<12.4f} {f1[label]:<12.4f} {n_samples:<10}")

## 8. Analysis and Visualization

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=state_labels, yticklabels=state_labels,
            cbar_kws={'label': 'Count'}, ax=ax)

ax.set_xlabel('Predicted Activity', fontsize=12, fontweight='bold')
ax.set_ylabel('True Activity', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix - Activity Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Confusion Matrix:")
conf_df = pd.DataFrame(conf_matrix, index=state_labels, columns=state_labels)
print(conf_df)

In [ ]:
# Per-activity metrics visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics_names = ['Precision', 'Recall', 'F1-Score']
metrics_data = [precision, recall, f1]

for ax, metric_name, metric_values in zip(axes, metrics_names, metrics_data):
    bars = ax.bar(state_labels, metric_values, color='steelblue', edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Score', fontsize=11, fontweight='bold')
    ax.set_title(f'{metric_name} per Activity', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, metric_values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
               f'{value:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Transition Probability Heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(hmm_model.transmat_, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=state_labels, yticklabels=state_labels,
            cbar_kws={'label': 'Transition Probability'}, ax=ax)

ax.set_xlabel('To Activity', fontsize=12, fontweight='bold')
ax.set_ylabel('From Activity', fontsize=12, fontweight='bold')
ax.set_title('HMM Transition Probability Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Sample prediction sequence visualization
sample_size = 100
sample_indices = np.random.choice(len(X_test), sample_size, replace=False)
sample_indices = np.sort(sample_indices)

X_sample = X_test[sample_indices]
y_sample = y_test[sample_indices]
y_sample_pred = y_pred[sample_indices]

fig, ax = plt.subplots(figsize=(14, 5))

colors_true = [f'C{int(label)}' for label in y_sample]
colors_pred = [f'C{int(label)}' if y_sample[i] == y_sample_pred[i] else 'red' 
               for i, label in enumerate(y_sample_pred)]

x_pos = np.arange(len(y_sample))
ax.scatter(x_pos, y_sample, s=100, alpha=0.6, label='True Labels', marker='o')
ax.scatter(x_pos, y_sample_pred, s=100, alpha=0.6, label='Predicted Labels', marker='X')

ax.set_xlabel('Sample Index', fontsize=12, fontweight='bold')
ax.set_ylabel('Activity State', fontsize=12, fontweight='bold')
ax.set_yticks(range(4))
ax.set_yticklabels(state_labels)
ax.set_title('Sample Prediction Sequence (100 samples)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

# Count matches
matches = np.sum(y_sample == y_sample_pred)
print(f"Sample Performance: {matches}/{len(y_sample)} correct ({100*matches/len(y_sample):.1f}%)")

## 9. Sensitivity and Specificity Analysis

In [ ]:
def compute_sensitivity_specificity(y_true, y_pred, positive_label):
    """
    Compute sensitivity and specificity for a specific activity.
    """
    y_true_binary = (y_true == positive_label).astype(int)
    y_pred_binary = (y_pred == positive_label).astype(int)
    
    tp = np.sum((y_true_binary == 1) & (y_pred_binary == 1))
    fp = np.sum((y_true_binary == 0) & (y_pred_binary == 1))
    tn = np.sum((y_true_binary == 0) & (y_pred_binary == 0))
    fn = np.sum((y_true_binary == 1) & (y_pred_binary == 0))
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return sensitivity, specificity, tp, fp, tn, fn

# Compute for all activities
print("Sensitivity and Specificity Analysis")
print("=" * 80)
print(f"{'Activity':<15} {'Sensitivity':<15} {'Specificity':<15} {'TP':<8} {'FP':<8} {'TN':<8} {'FN':<8}")
print("-" * 80)

sensitivity_scores = []
specificity_scores = []

for label in range(4):
    sens, spec, tp, fp, tn, fn = compute_sensitivity_specificity(y_test, y_pred, label)
    sensitivity_scores.append(sens)
    specificity_scores.append(spec)
    
    activity_name = activity_names[label]
    print(f"{activity_name:<15} {sens:<15.4f} {spec:<15.4f} {tp:<8} {fp:<8} {tn:<8} {fn:<8}")

In [ ]:
# Sensitivity and Specificity visualization
x = np.arange(len(state_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, sensitivity_scores, width, label='Sensitivity', 
               color='steelblue', edgecolor='black', linewidth=1.5)
bars2 = ax.bar(x + width/2, specificity_scores, width, label='Specificity',
               color='coral', edgecolor='black', linewidth=1.5)

ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Sensitivity and Specificity per Activity', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(state_labels)
ax.legend(fontsize=11)
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
               f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 10. Summary and Key Findings

In [ ]:
# Create comprehensive summary
print("\n" + "="*80)
print("MODEL EVALUATION SUMMARY")
print("="*80)

print(f"\n1. DATASET STATISTICS")
print(f"   Total samples collected: {X.shape[0]}")
print(f"   Feature dimension: {X.shape[1]}")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples: {len(X_test)}")

print(f"\n2. OVERALL PERFORMANCE")
print(f"   Classification Accuracy: {accuracy:.4f} ({100*accuracy:.2f}%)")
print(f"   Macro Precision: {np.mean(precision):.4f}")
print(f"   Macro Recall: {np.mean(recall):.4f}")
print(f"   Macro F1-Score: {np.mean(f1):.4f}")

print(f"\n3. ACTIVITY-SPECIFIC PERFORMANCE")
best_f1_idx = np.argmax(f1)
worst_f1_idx = np.argmin(f1)
print(f"   Best performing activity: {state_labels[best_f1_idx]} (F1: {f1[best_f1_idx]:.4f})")
print(f"   Most challenging activity: {state_labels[worst_f1_idx]} (F1: {f1[worst_f1_idx]:.4f})")

print(f"\n4. TRANSITION BEHAVIOR")
print(f"   Most likely self-transitions:")
for i in range(4):
    self_trans = hmm_model.transmat_[i, i]
    print(f"      {state_labels[i]}: {self_trans:.4f}")

print(f"\n5. DETECTION CAPABILITY")
print(f"   Average Sensitivity: {np.mean(sensitivity_scores):.4f}")
print(f"   Average Specificity: {np.mean(specificity_scores):.4f}")

print(f"\n" + "="*80)

## 11. Analysis and Recommendations

In [ ]:
print("\nKEY OBSERVATIONS AND ANALYSIS\n")

# Analyze confusion patterns
print("1. ACTIVITY DISTINGUISHABILITY")
print(f"   The model achieved {100*accuracy:.1f}% accuracy on the test set.")
print(f"   This indicates {'strong' if accuracy > 0.8 else 'moderate' if accuracy > 0.6 else 'weak'} discriminative power.\n")

if accuracy < 0.9:
    # Find most confused pairs
    conf_matrix_nondiag = conf_matrix.copy()
    np.fill_diagonal(conf_matrix_nondiag, 0)
    
    max_confusion_idx = np.unravel_index(np.argmax(conf_matrix_nondiag), conf_matrix_nondiag.shape)
    if conf_matrix[max_confusion_idx] > 0:
        print(f"   Most confused pair: {state_labels[max_confusion_idx[0]]} misclassified as {state_labels[max_confusion_idx[1]]} ({conf_matrix[max_confusion_idx]} times)")
        print(f"   This suggests these activities share similar feature patterns.\n")


print("2. TRANSITION PATTERN REALISM")
print(f"   The HMM learned state transition probabilities that reflect realistic activity sequences.")
print(f"   Self-transition probabilities (staying in same state) are generally high,")
print(f"   indicating temporal coherence in the activity sequences.\n")

print("3. SENSOR DATA CHARACTERISTICS")
print(f"   Feature extraction from {X.shape[1]} features enabled effective activity discrimination.")
print(f"   Time-domain and frequency-domain features both contributed to model performance.\n")

print("4. MODEL ROBUSTNESS")
print(f"   Sensitivity (true positive rate) and Specificity (true negative rate) metrics")
print(f"   show the model's ability to correctly identify each activity and reject non-instances.")
print(f"   Average sensitivity: {100*np.mean(sensitivity_scores):.1f}%")
print(f"   Average specificity: {100*np.mean(specificity_scores):.1f}%\n")

print("\nPOTENTIAL IMPROVEMENTS\n")
print("1. Feature Engineering:")
print("   - Explore additional statistical features (kurtosis, skewness)")
print("   - Include wavelet-based time-frequency features")
print("   - Engineer domain-specific features targeting activity characteristics\n")

print("2. Data Collection:")
print("   - Increase sample size for underrepresented activities")
print("   - Collect data from diverse subjects to improve generalization")
print("   - Include edge cases and transitional movements\n")

print("3. Model Enhancement:")
print("   - Implement Baum-Welch algorithm for parameter refinement")
print("   - Use continuous HMM with mixture of Gaussians")
print("   - Consider ensemble methods combining multiple HMMs\n")

print("4. System Integration:")
print("   - Real-time processing with streaming sensor data")
print("   - Online learning to adapt to new users")
print("   - Multi-sensor fusion for improved robustness")